## Propagation Model Comparison — LTE1800

This demo compares three propagation models applied to the same LTE1800 scenario (1800 MHz, urban environment):

- **FSPL** — Free Space Path Loss. Theoretical lower bound, assumes no obstacles or environmental effects.
- **Okumura-Hata** — Empirical model valid from 150–1500 MHz. Applied here at 1800 MHz, **outside its valid frequency range** — results are included for reference but should be interpreted with caution.
- **COST231-Hata** — Extension of Okumura-Hata valid from 1500–2000 MHz. The appropriate model for this scenario.

All system parameters are identical across the three simulations.

In [ ]:
import sys
import os
from src.run_simulation import run_simulation
sys.path.insert(0, os.path.abspath('..'))
from params.lte1800_config import lte1800
from src.utils.print_coverage import print_coverage
from dataclasses import replace
import numpy as np
import  plotly.graph_objects as go

In [2]:
distance = np.linspace(0.1,30,300)
lte1800_FSPL = replace(lte1800, name='lte1800_FSPL')
lte1800_Okumura = replace(lte1800, name='lte1800_Okumura')
lte1800_COST = replace(lte1800, name='lte1800_COST231')
[margin_lte1800_fspl, value0_lte1800_fspl,rx_lte1800_fspl] = run_simulation(lte1800,distance,'fspl',False)
[margin_lte1800_okumura, value0_lte1800_okumura,rx_lte1800_okumura] = run_simulation(lte1800,distance,'okumura-hata',False)
[margin_lte1800_cost231, value0_lte1800_cost231,rx_lte1800_cost231] = run_simulation(lte1800,distance,'cost231',False)

### Received Power vs Distance

FSPL produces the most optimistic estimate — no environmental losses beyond free-space propagation.

COST231 and Okumura-Hata introduce additional losses representing urban clutter and terrain effects.

The gap between FSPL and the empirical models illustrates the significant impact of the propagation

environment on real-world link performance.

In [3]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=distance, y=rx_lte1800_fspl, name="FSPL", line=dict(color="blue")))

fig.add_trace(go.Scatter(x=distance, y=rx_lte1800_okumura, name="Okumura-Hata", line=dict(color="red")))

fig.add_trace(go.Scatter(x=distance, y=rx_lte1800_cost231, name="COST231", line=dict(color="green")))
fig.update_layout(
    template='plotly_dark',
    title=dict(text ='Path Loss Comparison', x=0.5),
    xaxis=dict(range=[0, 30]), xaxis_title='Distance (km)',
    yaxis_title='Rx power (dBm)'
)

### Link Margin vs Distance

COST231 is the technically correct model for 1800 MHz in urban environments.

Okumura-Hata at this frequency exceeds its valid range and produces results that
may not accurately represent real propagation conditions.

The coverage distances reflect the different path loss assumptions of each model —
FSPL gives an upper bound on range, while COST231 provides the most realistic estimate.

In [4]:
print_coverage(lte1800_FSPL.name,value0_lte1800_fspl,distance)
print_coverage(lte1800_Okumura.name,value0_lte1800_okumura,distance)
print_coverage(lte1800_COST.name,value0_lte1800_cost231,distance)

fig = go.Figure()

fig.add_trace(go.Scatter(x=distance, y=margin_lte1800_fspl, name="FSPL", line=dict(color="blue")))

fig.add_trace(go.Scatter(x=distance, y=margin_lte1800_okumura, name="Okumura-Hata", line=dict(color="red")))

fig.add_trace(go.Scatter(x=distance, y=margin_lte1800_cost231, name="COST231", line=dict(color="green")))

fig.add_hline(
    y=0,
    line=dict(color='white', width=1.5, dash='dash'),
    annotation_text='Coverage threshold',
    annotation_position='bottom right'
)


fig.update_layout(
    template='plotly_dark',
    title=dict(text ='Path Loss Comparison', x=0.5),
    xaxis=dict(range=[0, 15]), xaxis_title='Distance (km)',
    yaxis_title='Link Margin (dB)',

)

lte1800_FSPL: Coverage available in all the range
lte1800_Okumura: Coverage available until 1.90 km
lte1800_COST231: Coverage available until 1.40 km


### Model Selection Considerations

Choosing the appropriate propagation model is critical in link budget analysis:

| Model | Valid Range | Use Case |
|-------|-------------|----------|
| FSPL | All frequencies | Theoretical baseline, LOS satellite links |
| Okumura-Hata | 150–1500 MHz | Urban/suburban terrestrial below 1500 MHz |
| COST231-Hata | 1500–2000 MHz | Urban/suburban terrestrial LTE bands |

For frequencies above 2000 MHz, additional models such as 3GPP TR 38.901 are required.